In [ ]:
import os
import rasterio as rio
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.path as mpth
from pathlib import Path


import Functions
import importlib

importlib.reload(Functions)

In [ ]:
app_path = Functions.get_input_path() / 'App'
output_folder = app_path / 'Documents'
input_path = app_path / 'Documents' / 'csv'

Collecting the backscatter data from a csv

In [ ]:
sigma_TM = pd.DataFrame(pd.read_csv(input_path / 'TimeSeries_sigma.csv'))
sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date'])
date_S1 = list(pd.unique(pd.to_datetime(sigma_TM['Date'])))
sigma_TM.drop(sigma_TM[sigma_TM['mean'] == 0].index,inplace=True)

sigma_TM = sigma_TM.sort_values(by=['Code', 'Band', 'Date'])

Fixed the dates of the moisture: shifted the dates of the moisture to the closest (previuos) S1 backscatter date to have the data at the same date.

In [ ]:
moisture = pd.read_excel(app_path / 'Data' / 'Ground_Campaign' / 'Flevoland_data' / 'Data_25_fields' / 'Average_Soil_moisture_N.xlsx',
                        header=0)

moisture.set_index('Code',inplace=True)
moisture = moisture[moisture.index.notna()]
moisture_date = list(pd.to_datetime(moisture.columns))

df_date_moist = pd.DataFrame({'original_date': moisture_date}).sort_values('original_date')
df_date_S1 = pd.DataFrame({'ref_date': date_S1}).sort_values('ref_date')

new_date_moist = pd.merge_asof(df_date_moist, df_date_S1, 
                                left_on='original_date',
                                right_on='ref_date', direction='backward')
moisture.columns = pd.to_datetime(new_date_moist['ref_date'])

moisture = moisture.reset_index().melt(id_vars='Code', var_name='Date', value_name='Moist_situ')


In [ ]:
sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date'])
moisture['Date'] = pd.to_datetime(moisture['Date'])

Try the aplha method: $$SSM_{i+1} \approx \frac{\sigma_{0}^{i+1}}{\sigma_{0}^{i}} SSM_{i}$$

In [ ]:
sigma_TM = pd.merge(
    sigma_TM, 
    moisture, 
    on=['Code', 'Date'], 
    how='right' 
)

calculated the difference between the backscatter at (t+1) and at (t). Then converted in linear scale. At the end grouped everything by the code of the field.

In [ ]:
sigma_TM.dropna(subset='Moist_situ' ,inplace=True)
sigma_TM['diff_mean'] = -sigma_TM.groupby(['Code', 'Band'])['mean'].diff(periods=-1)
sigma_TM['diff_lin'] = 10 ** (sigma_TM['diff_mean']/10)
sigma_TM.insert(len(sigma_TM.columns)-1, 'Moist_situ', sigma_TM.pop('Moist_situ'))

sigma_TM = sigma_TM.sort_values(by=['Code', 'Band', 'Date']).reset_index(drop=True)
display(sigma_TM[sigma_TM['Code'] == 1698168])

The alpha method: multiplied the ratio of backscatter (σ(t+1) / σ(t)). This ration is placed onthe same row of the SSM (t+1): for this reason, to calculate the SSM (t) we use ratio and the SSm at t-1

In [ ]:
def alpha_recursion(ssm_arr, ratio_arr, win=4):

    ssm_ret = ssm_arr.copy().astype(float)
    
    for b in range(0, len(ssm_ret), win):
        for passo in range(1, win):
    
            i = b + passo
            if i >= len(ssm_ret):
                break
            if not np.isnan(ssm_ret[i-1]) and not np.isnan(ratio_arr[i]):
                ssm_ret[i] = ratio_arr[i-1] * ssm_ret[i-1]
            else:
                continue
    
    return ssm_ret

sigma_TM['SSM_retrieved'] = sigma_TM.groupby(['Code', 'Band'], group_keys=False).apply(
    lambda g: pd.Series(alpha_recursion(g['Moist_situ'].values, g['diff_lin'].values), index=g.index)
)

In [ ]:
display(sigma_TM[sigma_TM['Code']==1698168])

As it mentions the article (Sentinel-1 Sensitivity to Soil Moisture at High Incidence Angle and the Impact on Retrieval Over Seasonal Crops, Davide Palmisano), the a priori datas were excluded from the linear regression: here they are excluded from the dataset.

In [ ]:
sigma_TM = sigma_TM.loc[sigma_TM['Moist_situ'] != sigma_TM['SSM_retrieved']]
display(sigma_TM)

Regression between the in-situ results and the SSM-retrieved with the alpha method for the single fields.

In [ ]:
from scipy import stats

results_regression = []

for (codice, banda, name,typ), gruppo in sigma_TM.groupby(['Code', 'Band', 'Name', 'Type']):   

    moist_situ = gruppo['Moist_situ'].values
    SSM_ret = gruppo['SSM_retrieved'].values

    if len(moist_situ) > 2:

        slope, intercept, r_value, p_value, std_err = stats.linregress(moist_situ, SSM_ret)        
        y_pred = slope * moist_situ + intercept
        
        residui = SSM_ret - y_pred
        
        rmsre = np.sqrt(np.mean(residui ** 2))
        
        results_regression.append({
            'Code': codice,
            'Name': name,
            'Band': banda,
            'Type': typ,
            'N_points': len(moist_situ),
            'R_Pearson': r_value,
            'R_2': r_value ** 2,
            'P_value': p_value,
            'Slope': slope,
            'Intercept': intercept,
            'RMSRE': rmsre
        })

df_statistiche = pd.DataFrame(results_regression)
df_statistiche.sort_values('R_2', ascending=False, inplace=True)
display(df_statistiche.head())

filename = "moisture_reg_xfield.csv"
output_path = output_folder / 'csv' / filename

df_statistiche.to_csv(output_path ,index=False, sep=';', decimal='.')

Here a new regression was calculaded, excluding the outliers (values that are 3*stdv away from the value found with the regression).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats
from matplotlib.backends.backend_pdf import PdfPages

results_regression_f = []

output_folder.mkdir(parents=True, exist_ok=True)

pdf_filename = output_folder / 'Regression_Moisture_xField.pdf'


with PdfPages(pdf_filename) as pdf:

    for i, (_, row) in enumerate(df_statistiche.iterrows()):

        codice = row['Code']
        banda = row['Band']
        tipo = row['Type']
        
        dati_gruppo = sigma_TM[(sigma_TM['Code'] == codice) & (sigma_TM['Band'] == banda)].sort_values('Date')
        
        x_in_situ = dati_gruppo['Moist_situ'].values
        y_calcolata = dati_gruppo['SSM_retrieved'].values
        date_asse = dati_gruppo['Date'].values
        
        if len(x_in_situ) > 2:
            
            # OUTLIER SEARCH
            y_pred_grezza = row['Slope'] * x_in_situ + row['Intercept']
            residui = y_calcolata - y_pred_grezza
            
            # THRESHOLD
            soglia = 3 * row['RMSRE']
            
            mask_outlier = np.abs(residui) > soglia
            mask_inlier = np.abs(residui) <= soglia
            n_outliers = np.sum(mask_outlier)
            
            x_puliti = x_in_situ[mask_inlier]
            y_puliti = y_calcolata[mask_inlier]
            
            if len(x_puliti) > 2:
                
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
                
                # REGRSSION W/ OUTLIER
                slope_f, intercept_f, r_f, p_f, std_err = stats.linregress(x_puliti, y_puliti)
                r2_pulito = r_f ** 2

                residui = y_puliti - x_puliti
                
                rmsre_f = np.sqrt(np.mean(residui ** 2))

                results_regression_f.append({
                    'Code': codice,
                    'Band': banda,
                    'Type': tipo,
                    'N_points': len(x_puliti),
                    'R_Pearson': r_f,
                    'R_2': r2_pulito,
                    'P_value': p_f,
                    'Slope': slope_f,
                    'Intercept': intercept_f,
                    'RMSRE': rmsre_f
                })

                # =================================================================
                # SUBPLOT 1: SCATTER PLOT & REGRESSIONE (ax1)
                # =================================================================
                
                # 1. Disegniamo i punti validi (Blue/Teal)
                sns.scatterplot(x=x_puliti, y=y_puliti, 
                                color='#1f77b4', s=60, label='Valid Data', alpha=0.8, ax=ax1)
                
                # 2. Disegniamo gli outlier rimossi (Rosso con 'X')
                if n_outliers > 0:
                    sns.scatterplot(x=x_in_situ[mask_outlier], y=y_calcolata[mask_outlier], 
                                    marker='x', color="#ca4828", s=60, label='Outliers', alpha=0.8, ax=ax1)
                
                # 3. Disegniamo la retta di regressione PULITA
                x_linea = np.linspace(x_puliti.min(), x_puliti.max(), 100)
                y_linea = slope_f * x_linea + intercept_f

                sns.lineplot(x=x_linea, y=y_linea, color="#ca4828", 
                             label=f'Cleaned Fit\n$R^2$ without outlier = {r2_pulito:.3f}', alpha=0.8, ax=ax1)
                
                ax1.fill_between(x_linea, 
                                 y_linea - std_err, 
                                 y_linea + std_err, 
                                 color="#ca4828", alpha=0.15, label='1 Std. Dev.')

                # Linea di riferimento ideale 1:1
                limiti = [min(x_in_situ.min(), y_calcolata.min()), max(x_in_situ.max(), y_calcolata.max())]
                ax1.plot(limiti, limiti, color='gray', linestyle=':', alpha=0.5, label='Ideal 1:1')
                
                # Formattazione pannello 1
                ax1.set_title(f"{codice} - {banda} - {tipo}\n($R^2$ raw: {row['R_2']:.3f})",
                              fontsize=11, fontweight='bold')
                ax1.set_xlabel("SSM In Situ ($m^3/m^3$)", fontsize=10)
                ax1.set_ylabel("SSM Retr ($m^3/m^3$)", fontsize=10)
                ax1.legend(loc='upper left', fontsize=9)
                ax1.grid(True, linestyle='--', alpha=0.5)

                # =================================================================
                # SUBPLOT 2: ANDAMENTO TEMPORALE (ax2)
                # =================================================================
                
                # Grafico linea + marker per l'umidità In Situ (usiamo il verde per distinguerlo bene)
                sns.lineplot(x=date_asse, y=x_in_situ, color='#2ca02c', marker='o', 
                             linewidth=1.8, label='SSM In Situ', ax=ax2)
                
                # Grafico linea + marker per l'umidità Stimata (riprendiamo il blu dei dati validi dello scatter)
                sns.lineplot(x=date_asse, y=y_calcolata, color='#1f77b4', marker='^', linestyle='--',
                             linewidth=1.5, label='SSM Retrieved', ax=ax2)
                
                # Opzionale ma consigliato: marchiamo con una 'X' rossa gli outlier anche sulla linea temporale della stima
                if n_outliers > 0:
                    ax2.scatter(date_asse[mask_outlier], y_calcolata[mask_outlier], 
                                marker='x', color="#ca4828", s=70, zorder=5, label='Outliers Identified')

                # Formattazione pannello 2
                ax2.set_title("Time Series Humidity", fontsize=11, fontweight='bold')
                ax2.set_xlabel("Date", fontsize=10)
                ax2.set_ylabel("SSM ($m^3/m^3$)", fontsize=10)
                ax2.legend(loc='upper right', fontsize=9)
                ax2.grid(True, linestyle='--', alpha=0.5)
                
                # Ruotiamo le date sull'asse X per non farle sovrapporre
                ax2.tick_params(axis='x', rotation=30)
                
                # Compattiamo il layout globale prima del salvataggio
                plt.tight_layout()
                
                # Salva la figura a due pannelli nella pagina corrente del PDF
                pdf.savefig()
                plt.close()




In [ ]:
results_regression_f = pd.DataFrame(results_regression_f)
results_regression_f.sort_values('R_2', ascending=False, inplace=True)
display(results_regression_f)

filename = "moisture_reg_xfield_no_outlier.csv"
output_path = output_folder / 'csv' / filename

results_regression_f.to_csv(output_path ,index=False, sep=';', decimal='.')

Here tha same thing is done but tthe regression was calculated on all the values of SSM available for the same type of field (eg wheat, grass, potatoes...).

In [ ]:
results_regression_group = []

for ( banda, typ ), gruppo in sigma_TM.groupby(['Band', 'Type']):   

    moist_situ = gruppo['Moist_situ'].values
    SSM_ret = gruppo['SSM_retrieved'].values

    if len(moist_situ) > 2:

        slope, intercept, r_value, p_value, std_err = stats.linregress(moist_situ, SSM_ret)        
        y_pred = slope * moist_situ + intercept
        
        residui = SSM_ret - y_pred
        
        rmsre = np.sqrt(np.mean(residui ** 2))
        
        results_regression_group.append({
            'Band': banda,
            'Type': typ,
            'N_points': len(moist_situ),
            'R_Pearson': r_value,
            'R_2': r_value ** 2,
            'P_value': p_value,
            'Slope': slope,
            'Intercept': intercept,
            'RMSRE': rmsre
        })

results_regression_group = pd.DataFrame(results_regression_group)
results_regression_group.sort_values('R_2', ascending=False, inplace=True)
display(results_regression_group)

filename = "moisture_reg_group.csv"
output_path = output_folder / 'csv' / filename

results_regression_group.to_csv(output_path ,index=False, sep=';', decimal='.')

In [ ]:
results_regression_group_f = []

pdf_filename = output_folder / 'Regression_Moisture_Group.pdf'

with PdfPages(pdf_filename) as pdf:

    for i, (_, row) in enumerate(results_regression_group.iterrows()):

        banda = row['Band']
        tipo = row['Type']
        
        dati_gruppo = sigma_TM[(sigma_TM['Type'] == tipo) & (sigma_TM['Band'] == banda)].sort_values('Date')
        
        x_in_situ = dati_gruppo['Moist_situ'].values
        y_calcolata = dati_gruppo['SSM_retrieved'].values
        date_asse = dati_gruppo['Date'].values
        
        if len(x_in_situ) > 2:
            
            # OUTLIER SEARCH
            y_pred_grezza = row['Slope'] * x_in_situ + row['Intercept']
            residui = y_calcolata - y_pred_grezza
            
            # THRESHOLD
            soglia = 3 * row['RMSRE']
            
            mask_outlier = np.abs(residui) > soglia
            mask_inlier = np.abs(residui) <= soglia
            n_outliers = np.sum(mask_outlier)
            
            x_puliti = x_in_situ[mask_inlier]
            y_puliti = y_calcolata[mask_inlier]
            
            if len(x_puliti) > 2:
                
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
                
                # REGRSSION W/ OUTLIER
                slope_f, intercept_f, r_f, p_f, std_err = stats.linregress(x_puliti, y_puliti)
                r2_pulito = r_f ** 2

                residui = y_puliti - x_puliti
                
                rmsre = np.sqrt(np.mean(residui ** 2))

                results_regression_group_f.append({
                    'Band': banda,
                    'Type': tipo,
                    'N_points': len(x_puliti),
                    'R_Pearson': r_f,
                    'R_2': r2_pulito,
                    'P_value': p_f,
                    'Slope': slope_f,
                    'Intercept': intercept_f,
                    'RMSRE': rmsre
                })

                # =================================================================
                # SUBPLOT 1: SCATTER PLOT & REGRESSIONE (ax1)
                # =================================================================
                
                # 1. Disegniamo i punti validi (Blue/Teal)
                sns.scatterplot(x=x_puliti, y=y_puliti, 
                                color='#1f77b4', s=60, label='Valid Data', alpha=0.8, ax=ax1)
                
                # 2. Disegniamo gli outlier rimossi (Rosso con 'X')
                if n_outliers > 0:
                    sns.scatterplot(x=x_in_situ[mask_outlier], y=y_calcolata[mask_outlier], 
                                    marker='x', color="#ca4828", s=60, label='Outliers', alpha=0.8, ax=ax1)
                
                # 3. Disegniamo la retta di regressione PULITA
                x_linea = np.linspace(x_puliti.min(), x_puliti.max(), 100)
                y_linea = slope_f * x_linea + intercept_f

                sns.lineplot(x=x_linea, y=y_linea, color="#ca4828", 
                             label=f'Cleaned Fit\n$R^2$ without outlier = {r2_pulito:.3f}', alpha=0.8, ax=ax1)
                
                ax1.fill_between(x_linea, 
                                 y_linea - std_err, 
                                 y_linea + std_err, 
                                 color="#ca4828", alpha=0.15, label='1 Std. Dev.')

                # Linea di riferimento ideale 1:1
                limiti = [min(x_in_situ.min(), y_calcolata.min()), max(x_in_situ.max(), y_calcolata.max())]
                ax1.plot(limiti, limiti, color='gray', linestyle=':', alpha=0.5, label='Ideal 1:1')
                
                # Formattazione pannello 1
                ax1.set_title(f"{tipo} - {banda}\n($R^2$ raw: {row['R_2']:.3f})",
                              fontsize=11, fontweight='bold')
                ax1.set_xlabel("SSM In Situ ($m^3/m^3$)", fontsize=10)
                ax1.set_ylabel("SSM Retr ($m^3/m^3$)", fontsize=10)
                ax1.legend(loc='upper left', fontsize=9)
                ax1.grid(True, linestyle='--', alpha=0.5)

                # =================================================================
                # SUBPLOT 2: ANDAMENTO TEMPORALE (ax2)
                # =================================================================
                
                # Grafico linea + marker per l'umidità In Situ (usiamo il verde per distinguerlo bene)
                sns.lineplot(x=date_asse, y=x_in_situ, color='#2ca02c', marker='o', 
                             linewidth=1.8, label='SSM In Situ', ax=ax2)
                
                # Grafico linea + marker per l'umidità Stimata (riprendiamo il blu dei dati validi dello scatter)
                sns.lineplot(x=date_asse, y=y_calcolata, color='#1f77b4', marker='^', linestyle='--',
                             linewidth=1.5, label='SSM Retrieved', ax=ax2)
                
                # Opzionale ma consigliato: marchiamo con una 'X' rossa gli outlier anche sulla linea temporale della stima
                if n_outliers > 0:
                    ax2.scatter(date_asse[mask_outlier], y_calcolata[mask_outlier], 
                                marker='x', color="#ca4828", s=70, zorder=5, label='Outliers Identified')

                # Formattazione pannello 2
                ax2.set_title("Time Series Humidity", fontsize=11, fontweight='bold')
                ax2.set_xlabel("Date", fontsize=10)
                ax2.set_ylabel("SSM ($m^3/m^3$)", fontsize=10)
                ax2.legend(loc='upper right', fontsize=9)
                ax2.grid(True, linestyle='--', alpha=0.5)
                
                # Ruotiamo le date sull'asse X per non farle sovrapporre
                ax2.tick_params(axis='x', rotation=30)
                
                # Compattiamo il layout globale prima del salvataggio
                plt.tight_layout()
                
                # Salva la figura a due pannelli nella pagina corrente del PDF
                pdf.savefig()
                plt.close()


In [ ]:
results_regression_group_f = pd.DataFrame(results_regression_group_f)
results_regression_group_f.sort_values('R_2', ascending=False, inplace=True)
display(results_regression_group_f)

filename = "moisture_reg_group_no_outlier.csv"
output_path = output_folder / 'csv' / filename

results_regression_group_f.to_csv(output_path ,index=False, sep=';', decimal='.')